In [ ]:
sc

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1763238347943_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<SparkContext master=yarn appName=livy-session-2>

In [ ]:
%%configure -f
{ "conf":{
        "spark.pyspark.python": "python",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type":"native",
        "spark.pyspark.virtualenv.bin.path":"/usr/bin/virtualenv"
       }
}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1763238347943_0004,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1763238347943_0004,pyspark,idle,Link,Link,None,✔


In [ ]:
# --- Install corrected packages ---
sc.install_pypi_package("numpy==1.23.5") 
sc.install_pypi_package("pandas==1.5.3") 
sc.install_pypi_package("scikit-learn==1.3.0") 
sc.install_pypi_package("shap==0.41.0") 
sc.install_pypi_package("matplotlib==3.7.1") 
# --- THIS IS THE FIX ---
# Use version 2.7.3, which is >=2.7 for matplotlib
# but old enough to work with pandas 1.5.3
sc.install_pypi_package("python-dateutil==2.7.3") 
# ---
sc.install_pypi_package("boto3==1.28.57") 
sc.install_pypi_package("seaborn")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Not uninstalling pandas at /usr/local/lib64/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1763238347943_0004/container_1763238347943_0004_01_000001/tmp/spark-4c8a1d37-739a-4d21-a45d-ae8b04ab6509
    Can't uninstall 'pandas'. No files were found to uninstall.



  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.9.4
    Not uninstalling matplotlib at /usr/local/lib64/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1763238347943_0004/container_1763238347943_0004_01_000001/tmp/spark-4c8a1d37-739a-4d21-a45d-ae8b04ab6509
    Can't uninstall 'matplotlib'. No files were found to uninstall.

  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Not uninstalling python-dateutil at /usr/lib/python3.9/site-packages, outside environment /mnt/yarn/usercache/

In [ ]:
sc.list_packages()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Package              Version
-------------------- -----------
appdirs              1.4.4
attrs                20.3.0
aws-cfn-bootstrap    2.0
awscli               2.30.4
awscrt               0.27.6
Babel                2.9.1
beautifulsoup4       4.9.3
boto                 2.49.0
boto3                1.28.57
botocore             1.31.85
cffi                 1.14.5
chardet              4.0.0
chevron              0.13.1
click                8.1.8
cloud-init           22.2.2
cloudpickle          3.1.2
colorama             0.4.4
configobj            5.0.6
contourpy            1.3.0
cryptography         36.0.1
cycler               0.12.1
dbus-python          1.2.18
distlib              0.3.1
distro               1.5.0
docutils             0.16
ec2-hibinit-agent    1.0.8
filelock             3.0.12
fonttools            4.60.1
gpg                  1.23.2
idna                 2.10
importlib_resources  6.5.2
Jinja2               2.11.3
jmespath             1.0.1
joblib               1.5.0
jsonpa

In [ ]:
import sys
import io
import boto3
import numpy as np
import pandas as pd
from typing import List, Tuple
from functools import reduce

from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# ML libraries
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.stat import Correlation

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# --- Set your S3 paths here ---
s3_processed_bucket = "s3://des431-hetero-agg-data/"

# Input path for processed features
feature_paths_to_test = [
    f"{s3_processed_bucket}/all_features_window_30s/"
]


# Output paths for all result CSVs
s3_results_path = "s3://des431-hetero-result/results_summary_csv/"
s3_importance_path = "s3://des431-hetero-result/feature_importance_summary_csv/"
s3_user_acc_path = "s3://des431-hetero-result/user_accuracy_summary_csv/"
s3_activity_acc_path = "s3://des431-hetero-result/activity_accuracy_summary_csv/"
s3_per_class_importance_path = "s3://des431-hetero-result/per_class_importance_summary_csv/"
s3_feature_log_path = "s3://des431-hetero-result/feature_selection_log_csv/"
s3_lodo_log_path = "s3://des431-hetero-result/lodo_fold_log_csv/"

# --- Path for saving correlation matrix data ---
s3_correlation_path = "s3://des431-hetero-result/correlation_matrix_csv/"

# --- Paths for Plotting and SHAP ---
s3_plot_path = "s3://des431-hetero-result/results_plots/"
s3_shap_data_path = "s3://des431-hetero-result/shap_data_prep_csv/"


# --- Correlation threshold for feature removal ---
CORR_THRESHOLD = 0.9

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
def perform_feature_selection(df: DataFrame, all_feature_cols: List[str], experiment_name: str, s3_log_path: str) -> Tuple[List[str], List[str]]:
    """
    Performs correlation analysis on features, removes highly correlated ones, 
    and logs the used/dropped features to S3.
    
    --- NEW: Also saves the full correlation matrix as a CSV. ---
    """
    print(f"--- Performing feature selection for {experiment_name} ---")
    print(f"Original feature count: {len(all_feature_cols)}")
    
    # 1. Calculate Correlation Matrix
    try:
        corr_assembler = VectorAssembler(inputCols=all_feature_cols, outputCol="corr_features", handleInvalid="skip")
        corr_df = corr_assembler.transform(df.select(all_feature_cols))
        
        if corr_df.rdd.isEmpty():
            print("WARNING: Correlation calculation skipped, VectorAssembler produced 0 rows.")
            return all_feature_cols, []

        corr_matrix = Correlation.corr(corr_df, "corr_features").collect()[0][0]
        
    except Exception as e:
        print(f"ERROR calculating correlation matrix: {e}. Using all features.")
        return all_feature_cols, []

    # --- NEW: Save Matrix to S3 as CSV ---
    try:
        print(f"Converting and saving correlation matrix to {s3_correlation_path}...")
        matrix_data = corr_matrix.toArray().tolist()
        
        # Create schema
        schema_cols = [StructField("feature_name", StringType())]
        for col_name in all_feature_cols:
            schema_cols.append(StructField(col_name, DoubleType()))
        corr_schema = StructType(schema_cols)
        
        # Prep data rows
        data_with_names = []
        for i, row in enumerate(matrix_data):
            row_name = all_feature_cols[i]
            data_with_names.append([row_name] + row)
            
        # Create Spark DF
        spark_corr_df = spark.createDataFrame(data_with_names, schema=corr_schema)
        
        # Save it as one CSV file
        s3_safe_name = experiment_name.replace(" ", "_").replace("/", "")
        csv_path = f"{s3_correlation_path}/{s3_safe_name}/"
        (spark_corr_df
            .coalesce(1) # Save as single file
            .write
            .mode("overwrite")
            .option("header", "true")
            .csv(csv_path)
        )
        print(f"Successfully saved correlation matrix to {csv_path}")
    except Exception as e:
        print(f"ERROR: Could not save correlation matrix. {e}")
    # --- END NEW SECTION ---

    # 2. Convert to Pandas for easier iteration (existing logic)
    matrix_pd = pd.DataFrame(corr_matrix.toArray(), columns=all_feature_cols, index=all_feature_cols)

    # 3. Identify features to drop (existing logic)
    features_to_drop = set()
    for i, col_name in enumerate(matrix_pd.columns):
        if col_name in features_to_drop:
            continue
        
        for j in range(i + 1, len(matrix_pd.columns)):
            row_name = matrix_pd.index[j]
            if row_name in features_to_drop:
                continue
            
            corr_value = matrix_pd.loc[row_name, col_name]
            
            if abs(corr_value) > CORR_THRESHOLD:
                features_to_drop.add(row_name) 
                
    features_used = [f for f in all_feature_cols if f not in features_to_drop]
    dropped_features = list(features_to_drop)

    print(f"Features used: {len(features_used)}, Features dropped: {len(dropped_features)}")

    # 4. Log the results to S3 (existing logic)
    try:
        log_data = [(experiment_name, "used", f) for f in features_used] + \
                   [(experiment_name, "dropped", f) for f in dropped_features]
        
        log_schema = StructType([
            StructField("experiment_name", StringType()),
            StructField("status", StringType()),
            StructField("feature_name", StringType())
        ])
        
        log_df = spark.createDataFrame(log_data, schema=log_schema)
        
        s3_safe_name = experiment_name.replace(" ", "_").replace("/", "")
        csv_path = f"{s3_feature_log_path}/{s3_safe_name}/"
        
        (log_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .option("header", "true")
            .csv(csv_path)
        )
        print(f"Successfully saved feature selection log to {csv_path}")
    
    except Exception as e:
        print(f"ERROR: Could not save feature log to S3. {e}")

    return features_used, dropped_features

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
def build_pipeline(model_type: str, features_used: List[str], label_col: str = "label") -> Pipeline:
    """
    Helper function to build a pipeline for a given model type.
    """
    
    label_indexer = StringIndexer(inputCol="gt_majority", outputCol=label_col, handleInvalid="skip")
    
    vector_assembler = VectorAssembler(inputCols=features_used, outputCol="features", handleInvalid="skip")
    
    if model_type == "RF":
        model_obj = RandomForestClassifier(labelCol=label_col, featuresCol="features", numTrees=100, seed=42)
        stages = [label_indexer, vector_assembler, model_obj]
        
    elif model_type == "LR":
        # Logistic Regression pipeline includes StandardScaler
        normalizer = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=True)
        model_obj = LogisticRegression(labelCol=label_col, featuresCol="scaledFeatures", maxIter=100)
        stages = [label_indexer, vector_assembler, normalizer, model_obj]
        
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
        
    return Pipeline(stages=stages)


def train_and_evaluate_80_20_split(train_df: DataFrame, test_df: DataFrame, features_used: List[str], experiment_name: str, model_type: str, label_col: str = "gt_majority") -> dict:
    """
    Trains and evaluates a model (RF or LR) on a pre-defined 80/20 split.
    This is the only function that generates feature importances.
    """
    print(f"\n--- Running 80/20 Experiment: {experiment_name} ({model_type}) ---")

    if train_df.rdd.isEmpty() or test_df.rdd.isEmpty():
        print(f"Skipping {experiment_name}: No training or testing data found.")
        return None
    
    train_count = train_df.count()
    test_count = test_df.count()
    print(f"Training rows: {train_count}, Testing rows: {test_count}")

    try:
        pipeline = build_pipeline(model_type, features_used, label_col="label")
    except Exception as e:
        print(f"Pipeline build failed: {e}")
        return None

    try:
        model = pipeline.fit(train_df)
    except Exception as e:
        print(f"Model training failed for {experiment_name}. Error: {e}")
        return None

    predictions = model.transform(test_df)
    
    evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    accuracy = evaluator_acc.evaluate(predictions)
    f1_score = evaluator_f1.evaluate(predictions)
    print(f"--- RESULT FOR {experiment_name} ({model_type}): Accuracy = {accuracy * 100:.2f}%, F1-Score = {f1_score * 100:.2f}% ---")

    predictions = predictions.withColumn("correct", F.when(F.col("label") == F.col("prediction"), 1).otherwise(0))
    user_accuracy_df = predictions.groupBy("User").agg(F.avg("correct").alias("user_accuracy"))
    user_accuracy_list = [(row.User, row.user_accuracy) for row in user_accuracy_df.collect()]
    activity_accuracy_df = predictions.groupBy(label_col).agg(F.avg("correct").alias("activity_accuracy"))
    activity_accuracy_list = [(row[label_col], row.activity_accuracy) for row in activity_accuracy_df.collect()]

    # Get Feature Importances
    feature_imp_list = None
    if model_type == "RF":
        print(f"--- Top 10 Feature Importances for {experiment_name} (RF) ---") 
        model_stage = model.stages[-1]
        importances = model_stage.featureImportances.toArray().tolist()
        feature_imp_list = sorted(list(zip(features_used, importances)), key=lambda x: x[1], reverse=True)
        
        for feature, imp in feature_imp_list[:10]: 
            print(f"{feature}: \t {imp * 100:.2f}%")
    
    return {
        "experiment_name": experiment_name,
        "model_type": model_type,
        "validation_type": "80_20_split",
        "accuracy": accuracy,
        "f1_score": f1_score,
        "feature_importances": feature_imp_list, 
        "user_accuracies": user_accuracy_list,
        "activity_accuracies": activity_accuracy_list
    }

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
def train_and_evaluate_louo_cv(df: DataFrame, features_used: List[str], experiment_name: str, model_type: str, label_col: str = "gt_majority") -> dict:
    """
    Performs Leave-One-User-Out (LOUO) cross-validation for a given model (RF or LR).
    """
    print(f"\n--- Running LOUO-CV Experiment: {experiment_name} ({model_type}) ---")

    if df.rdd.isEmpty():
        print(f"Skipping {experiment_name}: No data found.")
        return None

    users = [row.User for row in df.select("User").distinct().collect()]
    if len(users) < 2:
        print(f"Skipping {experiment_name}: Need at least 2 users for LOUO-CV.")
        return None
        
    print(f"Found {len(users)} users for LOUO-CV.")
    all_predictions_list = []

    for user_to_leave_out in users:
        print(f"  LOUO-CV: Holding out user {user_to_leave_out}...")
        
        test_df = df.filter(F.col("User") == user_to_leave_out)
        train_df = df.filter(F.col("User") != user_to_leave_out)
        
        if train_df.rdd.isEmpty() or test_df.rdd.isEmpty():
            print(f"  Skipping user {user_to_leave_out}: not enough data for split.")
            continue

        try:
            pipeline = build_pipeline(model_type, features_used, label_col="label")
            model = pipeline.fit(train_df)
            predictions = model.transform(test_df)
            all_predictions_list.append(predictions.select("label", "prediction", "User", label_col))
        except Exception as e:
            print(f"  Model training failed for user {user_to_leave_out}. Error: {e}")
            
    if not all_predictions_list:
        print("LOUO-CV failed: No predictions were generated.")
        return None

    all_predictions = reduce(DataFrame.unionAll, all_predictions_list)
    all_predictions.cache()

    print("Calculating final LOUO-CV metrics...")
    evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    accuracy = evaluator_acc.evaluate(all_predictions)
    f1_score = evaluator_f1.evaluate(all_predictions)
    print(f"--- RESULT FOR {experiment_name} ({model_type}): Accuracy = {accuracy * 100:.2f}%, F1-Score = {f1_score * 100:.2f}% ---")

    all_predictions = all_predictions.withColumn("correct", F.when(F.col("label") == F.col("prediction"), 1).otherwise(0))
    user_accuracy_df = all_predictions.groupBy("User").agg(F.avg("correct").alias("user_accuracy"))
    user_accuracy_list = [(row.User, row.user_accuracy) for row in user_accuracy_df.collect()]
    activity_accuracy_df = all_predictions.groupBy(label_col).agg(F.avg("correct").alias("activity_accuracy"))
    activity_accuracy_list = [(row[label_col], row.activity_accuracy) for row in activity_accuracy_df.collect()]
    all_predictions.unpersist()

    return {
        "experiment_name": experiment_name,
        "model_type": model_type,
        "validation_type": "louo_cv",
        "accuracy": accuracy,
        "f1_score": f1_score,
        "feature_importances": None,
        "user_accuracies": user_accuracy_list,
        "activity_accuracies": activity_accuracy_list
    }

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
def train_and_evaluate_lodo_cv(df: DataFrame, features_used: List[str], experiment_name: str, model_type: str, window_size: str, model_group: str, label_col: str = "gt_majority") -> Tuple[dict, List]:
    """
    Performs Leave-One-Device-Out (LODO) cross-validation for a given model (RF or LR).
    Returns the result dictionary AND a list of the fold logs.
    """
    print(f"\n--- Running LODO-CV Experiment: {experiment_name} ({model_type}) ---")

    # This list will store logs *only for this run*
    local_fold_log = []

    if df.rdd.isEmpty():
        print(f"Skipping {experiment_name}: No data found.")
        return None, local_fold_log

    devices = [row.Device for row in df.select("Device").distinct().collect()]
    if len(devices) < 2:
        print(f"Skipping {experiment_name}: Need at least 2 devices for LODO-CV (found: {devices}).")
        return None, local_fold_log
        
    print(f"Found {len(devices)} devices for LODO-CV: {devices}")
    all_predictions_list = []

    for device_to_leave_out in devices:
        train_devices = [d for d in devices if d != device_to_leave_out]
        train_devices_str = ",".join(train_devices)
        
        print(f"  LODO-CV: Hold out {device_to_leave_out}, Train on {train_devices_str}")
        
        test_df = df.filter(F.col("Device") == device_to_leave_out)
        train_df = df.filter(F.col("Device") != device_to_leave_out)
        
        if train_df.rdd.isEmpty() or test_df.rdd.isEmpty():
            print(f"  Skipping fold for {device_to_leave_out}: not enough data for split.")
            continue

        try:
            pipeline = build_pipeline(model_type, features_used, label_col="label")
            model = pipeline.fit(train_df)
            predictions = model.transform(test_df)
            
            # Log this fold's individual result
            eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
            fold_acc = eval_acc.evaluate(predictions)
            eval_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
            fold_f1 = eval_f1.evaluate(predictions)
            
            print(f"    -> Fold Result (Test on {device_to_leave_out}): Acc={fold_acc*100:.2f}%, F1={fold_f1*100:.2f}%")
            local_fold_log.append((model_group, train_devices_str, device_to_leave_out, fold_acc, fold_f1))

            all_predictions_list.append(predictions.select("label", "prediction", "User", "Device", label_col))
            
        except Exception as e:
            print(f"  Model training failed for fold {device_to_leave_out}. Error: {e}")
            
    if not all_predictions_list:
        print("LODO-CV failed: No predictions were generated.")
        return None, local_fold_log

    all_predictions = reduce(DataFrame.unionAll, all_predictions_list)
    all_predictions.cache()

    print("Calculating final aggregated LODO-CV metrics...")
    evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    accuracy = evaluator_acc.evaluate(all_predictions)
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    f1_score = evaluator_f1.evaluate(all_predictions)
    print(f"--- AGGREGATE RESULT FOR {experiment_name} ({model_type}): Accuracy = {accuracy * 100:.2f}%, F1-Score = {f1_score * 100:.2f}% ---")

    all_predictions = all_predictions.withColumn("correct", F.when(F.col("label") == F.col("prediction"), 1).otherwise(0))
    user_accuracy_df = all_predictions.groupBy("User").agg(F.avg("correct").alias("user_accuracy"))
    user_accuracy_list = [(row.User, row.user_accuracy) for row in user_accuracy_df.collect()]
    activity_accuracy_df = all_predictions.groupBy(label_col).agg(F.avg("correct").alias("activity_accuracy"))
    activity_accuracy_list = [(row[label_col], row.activity_accuracy) for row in activity_accuracy_df.collect()]
    all_predictions.unpersist()

    result_dict = {
        "experiment_name": experiment_name,
        "model_type": model_type,
        "validation_type": "lodo_cv",
        "accuracy": accuracy,
        "f1_score": f1_score,
        "feature_importances": None,
        "user_accuracies": user_accuracy_list,
        "activity_accuracies": activity_accuracy_list
    }
    
    return result_dict, local_fold_log

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def save_results_plot_to_s3(all_results_pandas_df: pd.DataFrame, window: str, s3_plot_path: str):
    """
    Generates a comparison bar plot for a given window's results
    and uploads it directly to S3.
    """
    print(f"\n--- Generating plot for {window} window ---")
    
    # 1. Setup S3 connection
    s3_bucket = s3_plot_path.split("/")[2]
    s3_key_prefix = "/".join(s3_plot_path.split("/")[3:])
    s3_key = f"{s3_key_prefix}{window}_comparison.png"
    s3 = boto3.client('s3')

    # 2. Filter for the current window and create a readable experiment name
    plot_df = all_results_pandas_df[all_results_pandas_df['window_size'] == window].copy()
    if plot_df.empty:
        print(f"No results to plot for {window}.")
        return

    plot_df['Experiment'] = plot_df['experiment_type'] + " | " + \
                            plot_df['model_group'] + " | " + \
                            plot_df['model_type']
    
    # 3. Generate Plot
    plt.figure(figsize=(16, 10))
    ax = sns.barplot(
        data=plot_df, 
        x='accuracy', 
        y='Experiment', 
        hue='validation_type', 
        palette="viridis"
    )
    # Convert accuracy (0.0-1.0) to percentage for the x-axis
    ax.set_xticks(ax.get_xticks()) # Get current tick locations
    ax.set_xticklabels([f'{x*100:.0f}%' for x in ax.get_xticks()]) # Format as %
    
    ax.set_title(f'Model Accuracy Comparison for {window} Window', fontsize=16)
    ax.set_xlabel('Accuracy', fontsize=12)
    ax.set_ylabel('Experiment', fontsize=12)
    plt.legend(title='Validation Type', loc='lower right')
    plt.tight_layout()
    
    # 4. Save plot to an in-memory buffer
    img_data = io.BytesIO()
    plt.savefig(img_data, format='png')
    img_data.seek(0)
    plt.close() # Close the plot to free memory

    # 5. Upload buffer to S3
    s3.put_object(Bucket=s3_bucket, Key=s3_key, Body=img_data)
    print(f"Successfully saved plot to s3://{s3_bucket}/{s3_key}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# --- Schemas for our incremental save DataFrames ---
results_schema = StructType([
    StructField("experiment_type", StringType(), True),
    StructField("model_group", StringType(), True),
    StructField("model_type", StringType(), True),
    StructField("validation_type", StringType(), True),
    StructField("accuracy", DoubleType(), True),
    StructField("f1_score", DoubleType(), True)
])

importance_schema = StructType([
    StructField("experiment_type", StringType(), True),
    StructField("model_group", StringType(), True),
    StructField("feature_name", StringType(), True),
    StructField("importance_score", DoubleType(), True)
])

user_acc_schema = StructType([
    StructField("experiment_type", StringType(), True),
    StructField("model_group", StringType(), True),
    StructField("user", StringType(), True),
    StructField("user_accuracy", DoubleType(), True)
])

activity_acc_schema = StructType([
    StructField("experiment_type", StringType(), True),
    StructField("model_group", StringType(), True),
    StructField("activity", StringType(), True),
    StructField("activity_accuracy", DoubleType(), True)
])

per_class_schema = StructType([
    StructField("activity_class", StringType(), True),
    StructField("feature_name", StringType(), True),
    StructField("importance_score", DoubleType(), True)
])

lodo_log_schema = StructType([
    StructField("model_group", StringType(), True),
    StructField("train_devices", StringType(), True),
    StructField("test_device", StringType(), True),
    StructField("fold_accuracy", DoubleType(), True),
    StructField("fold_f1_score", DoubleType(), True)
])

# --- Helper function to process results within the loop ---
def process_result(res, experiment_type, model_name, lists_dict):
    """
    Flattens a result dict and appends to the *local* window lists.
    """
    if not res:
        print(f"  Skipping result processing - No result provided.")
        return

    print(f"  > Processing result: {experiment_type} / {model_name} / {res['validation_type']}")
    
    lists_dict['flat_results_list'].append((
        experiment_type,
        model_name,
        res['model_type'], 
        res['validation_type'],
        res['accuracy'], 
        res['f1_score']
    ))
    
    if res['feature_importances']:
        for f, imp in res['feature_importances']:
            lists_dict['flat_importance_list'].append((experiment_type, model_name, f, imp))

    if res['user_accuracies']:
        for user, acc in res['user_accuracies']:
            lists_dict['flat_user_acc_list'].append((experiment_type, model_name, user, acc))
            
    if res['activity_accuracies']:
        for activity, acc in res['activity_accuracies']:
            lists_dict['flat_activity_acc_list'].append((experiment_type, model_name, activity, acc))


# --- Main Loop ---
for s3_features_path in feature_paths_to_test:
    window_size = s3_features_path.split('_')[-1].replace('s/', '')
    print(f"\n=======================================================")
    print(f"STARTING ALL EXPERIMENTS FOR: {window_size}s WINDOW")
    print(f"=======================================================")
    
    # --- These lists will hold results for THIS window ONLY ---
    window_lists = {
        'flat_results_list': [],
        'flat_importance_list': [],
        'flat_user_acc_list': [],
        'flat_activity_acc_list': [],
        'flat_per_class_list': [],
        'flat_lodo_log_list': []
    }

    # 1. Read features
    try:
        features_df = spark.read.parquet(s3_features_path)
    except Exception as e:
        print(f"Could not read path: {s3_features_path}. Skipping this window.")
        print(f"Error: {e}")
        continue

    # 2. Preprocessing
    features_df = features_df.withColumn("model", F.split(F.col("Device"), "_").getItem(0))
    phone_df = features_df.filter(F.col("source_type") == "phone")
    phone_df_cleaned_labels = phone_df.dropna(subset=["gt_majority"])
    sensor_feature_cols = [col for col in phone_df_cleaned_labels.columns if col.startswith('ac_') or col.startswith('gr_')]
    phone_df_cleaned = phone_df_cleaned_labels.dropna(subset=sensor_feature_cols)
    
    if phone_df_cleaned.rdd.isEmpty():
        print(f"No non-null phone data found for {window_size}s window. Skipping.")
        continue
        
    # 3. Feature Selection
    exp_name_base = f"{window_size}s_Sensor_Features"
    (features_used_sensors, dropped_sensors) = perform_feature_selection(
        phone_df_cleaned, sensor_feature_cols, exp_name_base, s3_feature_log_path
    )
    
    base_cols = ["User", "gt_majority", "model", "Device", "source_type"]
    phone_df_selected = phone_df_cleaned.select(base_cols + features_used_sensors).cache()
    
    # ==============================================================
    # 4. RUN GENERALIZED 80/20 & SHAP PREP
    # ==============================================================
    print("\n--- Splitting data for 80/20 GENERALIZED Phone Model ---")
    train_all_df, test_all_df = phone_df_selected.randomSplit([0.8, 0.2], seed=42)
    exp_name_gen = f"{window_size}s - Generalized Phone"

    # --- NEW: Save data for SHAP analysis as CSV ---
    print(f"--- Saving SHAP data for {window_size}s window ---")
    try:
        # Note: We save as a single CSV file for easier local processing
        shap_train_path = f"{s3_shap_data_path}/window_size={window_size}/train/"
        shap_test_path = f"{s3_shap_data_path}/window_size={window_size}/test/"
        label_col = "gt_majority"
        
        (train_all_df.select(features_used_sensors + [label_col])
            .coalesce(1) # Combine into 1 partition
            .write.mode("overwrite").option("header", "true").csv(shap_train_path)
        )
        (test_all_df.select(features_used_sensors + [label_col])
            .coalesce(1) # Combine into 1 partition
            .write.mode("overwrite").option("header", "true").csv(shap_test_path)
        )
        
        print(f"Successfully saved SHAP data to {s3_shap_data_path}/")
    except Exception as e:
        print(f"ERROR: Could not save SHAP data. {e}")
    
    # Run 80/20 experiment (for baseline and feature importance)
    res_80_20_rf = train_and_evaluate_80_20_split(
        train_all_df, test_all_df, features_used_sensors, exp_name_gen, "RF"
    )
    process_result(res_80_20_rf, "Generalized", "All_Phones", window_lists)
    
    res_80_20_lr = train_and_evaluate_80_20_split(
        train_all_df, test_all_df, features_used_sensors, exp_name_gen, "LR"
    )
    process_result(res_80_20_lr, "Generalized", "All_Phones", window_lists)

    # ==============================================================
    # 5. RUN GENERALIZED LOUO-CV
    # ==============================================================
    res_louo_rf = train_and_evaluate_louo_cv(
        phone_df_selected, features_used_sensors, exp_name_gen, "RF"
    )
    process_result(res_louo_rf, "Generalized", "All_Phones", window_lists)
    
    res_louo_lr = train_and_evaluate_louo_cv(
        phone_df_selected, features_used_sensors, exp_name_gen, "LR"
    )
    process_result(res_louo_lr, "Generalized", "All_Phones", window_lists)

    # ==============================================================
    # 6. RUN SPECIFIC MODEL EXPERIMENTS (LODO and LOUO)
    # ==============================================================
    model_list = [row.model for row in phone_df_selected.select("model").distinct().collect()]
    print(f"\nFound models for LODO & LOUO testing: {model_list}")
    
    for model_name in model_list:
        print(f"\n--- Starting Specific-Model tests for: {model_name} ---")
        model_df = phone_df_selected.filter(F.col("model") == model_name)

        # --- Test 1: LODO-CV ---
        exp_name_lodo = f"{window_size}s - Specific LODO ({model_name})"
        res_lodo_rf, folds_lodo_rf = train_and_evaluate_lodo_cv(
            model_df, features_used_sensors, exp_name_lodo, "RF", window_size, model_name
        )
        process_result(res_lodo_rf, "Specific LODO", model_name, window_lists)
        window_lists['flat_lodo_log_list'].extend(folds_lodo_rf)

        res_lodo_lr, folds_lodo_lr = train_and_evaluate_lodo_cv(
            model_df, features_used_sensors, exp_name_lodo, "LR", window_size, model_name
        )
        process_result(res_lodo_lr, "Specific LODO", model_name, window_lists)
        window_lists['flat_lodo_log_list'].extend(folds_lodo_lr)

        # --- Test 2: LOUO-CV ---
        exp_name_louo = f"{window_size}s - Specific LOUO ({model_name})"
        res_spec_louo_rf = train_and_evaluate_louo_cv(
            model_df, features_used_sensors, exp_name_louo, "RF"
        )
        process_result(res_spec_louo_rf, "Specific LOUO", model_name, window_lists)
        
        res_spec_louo_lr = train_and_evaluate_louo_cv(
            model_df, features_used_sensors, exp_name_louo, "LR"
        )
        process_result(res_spec_louo_lr, "Specific LOUO", model_name, window_lists)
        
    # ==============================================================
    # 7. RUN Per-Class Importance (One-vs-Rest)
    # ==============================================================
    print(f"\n--- RUNNING EXPERIMENT 7: Per-Class Importance ({window_size}s) ---")
    activity_list = [row.gt_majority for row in phone_df_selected.select("gt_majority").distinct().collect()]
    
    for activity in activity_list:
        # (Rest of Per-Class logic is unchanged...)
        ovr_df = phone_df_selected.withColumn("ovr_label", F.when(F.col("gt_majority") == activity, 1).otherwise(0))
        train_ovr_df, test_ovr_df = ovr_df.randomSplit([0.8, 0.2], seed=42)
        vector_assembler = VectorAssembler(inputCols=features_used_sensors, outputCol="features", handleInvalid="skip")
        rf_binary = RandomForestClassifier(labelCol="ovr_label", featuresCol="features", numTrees=50, seed=42)
        pipeline_binary = Pipeline(stages=[vector_assembler, rf_binary])
        
        try:
            binary_model = pipeline_binary.fit(train_ovr_df)
            rf_model_binary = binary_model.stages[-1]
            importances = rf_model_binary.featureImportances
            feature_imp_list = sorted(list(zip(features_used_sensors, importances.toArray().tolist())), key=lambda x: x[1], reverse=True)
            print(f"    Top 5 features for '{activity}': {[f[0] for f in feature_imp_list[:5]]}")
            for f, imp in feature_imp_list:
                window_lists['flat_per_class_list'].append((activity, f, imp))
        except Exception as e:
            print(f"    Model training failed for OvR '{activity}'. Error: {e}")
            
    phone_df_selected.unpersist()

    # ==============================================================
    # 8. --- NEW: WRITE ALL RESULTS FOR THIS WINDOW ---
    # ==============================================================
    print(f"\n--- Writing all results for {window_size}s window to S3 as CSV ---")
    
    def write_partitioned_data(data_list, schema, s3_path):
        if data_list:
            df = spark.createDataFrame(data_list, schema=schema)
            # Add the window_size column for partitioning
            df = df.withColumn("window_size", F.lit(window_size))
            
            # --- MODIFIED: Write as CSV ---
            (df.write
               .partitionBy("window_size")
               .mode("append") # append mode skips if partition exists
               .option("header", "true") # Add header to CSVs
               .csv(s3_path) 
            )
            print(f"Successfully appended data to {s3_path}")
        else:
            print(f"No data to write for {s3_path}")

    try:
        # Write all 6 files as CSV
        write_partitioned_data(window_lists['flat_results_list'], results_schema, s3_results_path)
        write_partitioned_data(window_lists['flat_importance_list'], importance_schema, s3_importance_path)
        write_partitioned_data(window_lists['flat_user_acc_list'], user_acc_schema, s3_user_acc_path)
        write_partitioned_data(window_lists['flat_activity_acc_list'], activity_acc_schema, s3_activity_acc_path)
        write_partitioned_data(window_lists['flat_per_class_list'], per_class_schema, s3_per_class_importance_path)
        write_partitioned_data(window_lists['flat_lodo_log_list'], lodo_log_schema, s3_lodo_log_path)

        print(f"--- S3 Write Complete for {window_size}s ---")
        
    except Exception as e:
        print(f"\nERROR: Could not save results to S3 for {window_size}s.")
        print(e)

print("\n\n================================================")
print("===== ALL EXPERIMENT WINDOWS COMPLETE =====")
print("================================================")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


STARTING ALL EXPERIMENTS FOR: 1s WINDOW
--- Performing feature selection for 1s_Sensor_Features ---
Original feature count: 64
Converting and saving correlation matrix to s3://des431-hetero-result/correlation_matrix_csv/...
Successfully saved correlation matrix to s3://des431-hetero-result/correlation_matrix_csv//1s_Sensor_Features/
Features used: 43, Features dropped: 21
Successfully saved feature selection log to s3://des431-hetero-result/feature_selection_log_csv//1s_Sensor_Features/

--- Splitting data for 80/20 GENERALIZED Phone Model ---
--- Saving SHAP data for 1s window ---
Successfully saved SHAP data to s3://des431-hetero-result/shap_data_prep_csv//

--- Running 80/20 Experiment: 1s - Generalized Phone (RF) ---
Training rows: 65534, Testing rows: 16156
--- RESULT FOR 1s - Generalized Phone (RF): Accuracy = 73.36%, F1-Score = 68.43% ---
--- Top 10 Feature Importances for 1s - Generalized Phone (RF) ---
ac_x_min: 	 11.04%
ac_x_max: 	 9.43%
ac_x_std: 	 9.40%
gr_x_energy: 	 9.35

In [ ]:
print("\n\n================================================")
print("===== FINAL STEP: READING ALL DATA AND PLOTTING =====")
print("================================================")

try:
    # 1. Read all partitioned results back from S3
    print(f"Reading all CSV results from {s3_results_path}...")
    
    # --- MODIFIED: Read from CSV ---
    # We must also specify the schema to read it back correctly
    final_results_schema = results_schema.add("window_size", StringType(), True)
    
    all_results_df = spark.read \
        .option("header", "true") \
        .schema(final_results_schema) \
        .csv(s3_results_path)
    
    # Check if we have any data to plot
    if all_results_df.rdd.isEmpty():
        print("No results found to plot.")
    else:
        # Cast accuracy column (it's already DoubleType due to schema)
        print("Successfully read results. Converting to Pandas for plotting...")
        all_results_pd = all_results_df.toPandas()
        
        # Get a list of all unique windows we have results for
        windows_to_plot = all_results_pd['window_size'].unique()
        print(f"Found results for windows: {windows_to_plot}")

        # 2. Generate and save plots for all windows
        for window in windows_to_plot:
            try:
                save_results_plot_to_s3(all_results_pd, window, s3_plot_path)
            except Exception as e:
                print(f"ERROR: Could not generate plot for {window}. {e}")
        
    print("Plotting complete.")

except Exception as e:
    print(f"\nERROR: Could not read CSV results from S3 for plotting.")
    print(f"Path: {s3_results_path}")
    print(e)

print("\nJob complete.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…



===== FINAL STEP: READING ALL DATA AND PLOTTING =====
Reading all CSV results from s3://des431-hetero-result/results_summary_csv/...
Successfully read results. Converting to Pandas for plotting...
Found results for windows: ['1' '10' '20' '5' '3']

--- Generating plot for 1 window ---
Successfully saved plot to s3://des431-hetero-result/results_plots/1_comparison.png

--- Generating plot for 10 window ---
Successfully saved plot to s3://des431-hetero-result/results_plots/10_comparison.png

--- Generating plot for 20 window ---
Successfully saved plot to s3://des431-hetero-result/results_plots/20_comparison.png

--- Generating plot for 5 window ---
Successfully saved plot to s3://des431-hetero-result/results_plots/5_comparison.png

--- Generating plot for 3 window ---
Successfully saved plot to s3://des431-hetero-result/results_plots/3_comparison.png
Plotting complete.

Job complete.